# LoFi-MedG HER2 — Arm A on a free Colab GPU

Runs the IHC track end to end: BCI → dataset → smoke test → cached features →
fine-tune → evaluate. Written for a free T4 (16 GB).

**Read before running:**

- Cells are ordered and each one checks its own preconditions. Run them in order.
- Free Colab disconnects. Section 2 mounts Drive and everything heavy is written
  there, so a disconnect costs the current epoch, not the whole run.
- Two things need *your* account and cannot be automated: accepting the
  `google/gemma-3-270m-it` licence, and obtaining the BCI download link. Both are
  flagged inline.
- The BCI track score is **not** a HER2 detection result — its targets come from
  our own DAB threshold. See `RESULTS_her2.md` §6, which fixes that reading in
  advance.

## 1. Confirm a GPU is attached

If this fails: Runtime → Change runtime type → T4 GPU.

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi produced no output')

import torch
assert torch.cuda.is_available(), 'No CUDA device. Runtime -> Change runtime type -> T4 GPU.'
props = torch.cuda.get_device_properties(0)
print(f'{props.name}, {props.total_memory / 1024**3:.1f} GB, torch {torch.__version__}')

## 2. Persist to Drive

Checkpoints, the dataset and the feature cache all go here so a disconnect is
recoverable. Budget roughly: BCI ~10 GB, converted dataset ~2 GB, feature cache
~0.6 KB × 1024 per 1000 images (see §5).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/lofi_her2'
os.makedirs(WORK, exist_ok=True)
for sub in ('data', 'models', 'cache', 'results'):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
print('Working directory:', WORK)
print(subprocess.run(['df', '-h', WORK], capture_output=True, text=True).stdout)

## 3. Get the code

The HER2 extension lives on the `her2-extension` branch of your own clone. Pick
whichever applies:

- **A.** You pushed it to your own GitHub fork → set `REPO_URL`.
- **B.** You have not pushed it anywhere → zip the local repo, upload the zip to
  Drive at `lofi_her2/lofi-medg.zip`, and leave `REPO_URL` empty.

Do not point this at `myeongkyunkang/lofi-medg` — upstream does not contain the
HER2 extension.

In [ ]:
REPO_URL = ''          # e.g. 'https://github.com/<you>/lofi-medg.git'
BRANCH = 'her2-extension'
REPO = '/content/lofi-medg'

if os.path.isdir(REPO):
    print('Already present at', REPO)
elif REPO_URL:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO], check=True)
else:
    zip_path = os.path.join(WORK, 'lofi-medg.zip')
    assert os.path.isfile(zip_path), (
        f'Set REPO_URL, or upload the repo zip to {zip_path}. '
        f'On Windows: Compress-Archive -Path lofi-medg -DestinationPath lofi-medg.zip'
    )
    subprocess.run(['unzip', '-q', zip_path, '-d', '/content/'], check=True)

assert os.path.isfile(os.path.join(REPO, 'main.py')), f'{REPO} does not look like the repo'
os.chdir(REPO)
sys.path.insert(0, REPO)
print('Repo ready at', REPO)

In [ ]:
# Dependencies. Colab already ships torch/numpy/pandas; this fills the gaps.
!pip install -q transformers tokenizers safetensors huggingface_hub pillow scikit-image tqdm matplotlib

# Sanity: the suite runs without data, a GPU or the checkpoints. If this fails,
# the upload is incomplete -- stop here rather than debugging later on GPU time.
!python -m pytest tests -q

## 4. Checkpoints

**Manual step first:** open https://huggingface.co/google/gemma-3-270m-it and
accept the licence with the same account as the token below. The download fails
with a 401/403 until you do. The other two repositories are ungated.

In [ ]:
from huggingface_hub import login, snapshot_download
from google.colab import userdata

# Store the token in Colab Secrets (key icon, left sidebar) as HF_TOKEN.
# Pasting it into a cell puts it in the saved notebook -- do not.
login(token=userdata.get('HF_TOKEN'))

MODELS = os.path.join(WORK, 'models')

targets = {
    'google/gemma-3-270m-it': 'gemma-3-270m-it',        # GATED: accept the licence first
    'google/siglip2-so400m-patch16-512': 'siglip2-so400m-patch16-512',
    'myeongkyunkang/lofi-medg': 'lofi-medg',
}
for repo_id, local_name in targets.items():
    target = os.path.join(MODELS, local_name)
    if os.path.isdir(target) and os.listdir(target):
        print(f'skip {repo_id} (already present)')
        continue
    print(f'downloading {repo_id} ...')
    snapshot_download(repo_id=repo_id, local_dir=target)

for name in targets.values():
    print(name, '->', sorted(os.listdir(os.path.join(MODELS, name)))[:6])

## 5. BCI data

**Manual step:** BCI is distributed via links on https://bupt-ai-cz.github.io/BCI/
(Google Drive / Baidu), and those links change. Fetch the current one yourself and
set `BCI_URL`, or download it separately and place the archive at
`lofi_her2/data/BCI_dataset.zip` on Drive.

I have deliberately not hard-coded a URL here — a stale or guessed link that
silently fetches the wrong thing is worse than an explicit stop.

Expected layout after extraction:

```
data/BCI_dataset/IHC/train/<id>_train_<grade>.png
data/BCI_dataset/IHC/test/<id>_test_<grade>.png     grade in {0, 1+, 2+, 3+}
```

In [ ]:
BCI_URL = ''   # paste the current link from the BCI project page, or leave empty
BCI_DIR = os.path.join(WORK, 'data', 'BCI_dataset')
archive = os.path.join(WORK, 'data', 'BCI_dataset.zip')

if not os.path.isdir(os.path.join(BCI_DIR, 'IHC')):
    if not os.path.isfile(archive):
        assert BCI_URL, (
            'No BCI archive and no BCI_URL. Get the current download link from '
            'https://bupt-ai-cz.github.io/BCI/ and set BCI_URL, or upload the zip to '
            f'{archive}'
        )
        !pip install -q gdown
        import gdown; gdown.download(BCI_URL, archive, quiet=False, fuzzy=True)
    subprocess.run(['unzip', '-q', archive, '-d', os.path.join(WORK, 'data')], check=True)

ihc = os.path.join(BCI_DIR, 'IHC')
assert os.path.isdir(ihc), f'expected {ihc}; check the extracted layout'
for split in ('train', 'test'):
    files = os.listdir(os.path.join(ihc, split))
    print(f'{split}: {len(files)} patches, e.g. {sorted(files)[:2]}')

## 6. Build the HER2 dataset

DAB deconvolution → boxes → templated captions. The boxes are a **heuristic**,
not annotation; `her2_meta.csv` records `content_type` per sample so evaluation
can be split by supervision strength.

Set `LIMIT` to a few hundred for a fast first pass, then rerun with `0` for the
real dataset.

In [ ]:
HER2_DIR = os.path.join(WORK, 'data', 'her2_512p')
LIMIT = 0   # 0 = all; e.g. 200 for a quick pass

cmd = ['python', 'tools/preprocess_bci.py', '--bci_dir', BCI_DIR, '--output_dir', HER2_DIR]
if LIMIT:
    cmd += ['--limit_per_split', str(LIMIT)]
subprocess.run(cmd, check=True)

subprocess.run(['python', 'tools/merge_her2_manifests.py', '--output_dir', HER2_DIR], check=True)

import pandas as pd
meta = pd.read_csv(os.path.join(HER2_DIR, 'her2_meta.csv'))
print(meta['content_type'].value_counts(), '\n')
print(meta.head())

## 7. Smoke test

A few samples, one epoch, real checkpoints. Proves the wiring before anything
expensive. **The loss it prints is meaningless** — it must not be recorded.

In [ ]:
ENCODER = 'siglip2-so400m-patch16-512'
RESUME = os.path.join(MODELS, 'lofi-medg', 'last.pt')
assert os.path.isfile(RESUME), f'released checkpoint not found at {RESUME}'

subprocess.run([
    'python', 'tools/smoke_test_her2.py',
    '--her2_dir', HER2_DIR, '--model_dir', MODELS, '--model_name', ENCODER,
    '--resume', RESUME, '--result_dir', os.path.join(WORK, 'results'),
    '--feature_cache_dir', os.path.join(WORK, 'cache', 'smoke'),
    '--with_cache',
], check=True)

## 8. Precompute frozen-encoder features

`--fix_enc` freezes the encoder and the pipeline applies no augmentation, so the
encoder output per image is identical across all 30 epochs. Computing it once
removes the dominant cost. This is exact, not an approximation — see
`README_her2.md` §7b.

Run once; it is resumable if the session drops.

In [ ]:
CACHE = os.path.join(WORK, 'cache', 'her2_features')

subprocess.run([
    'python', 'tools/precompute_features.py',
    '--dataset', 'her2', '--her2_dir', HER2_DIR,
    '--splits', 'train', 'val', 'test',
    '--model_dir', MODELS, '--model_name', ENCODER, '--resume', RESUME,
    '--pool2x2', '--feature_cache_dir', CACHE,
    '--multimodal_tokens', '128', '--decoder_max_length', '200',
    '--batch_size', '16',
], check=True)

## 9. Arm A — fine-tune from the released checkpoint

Frozen encoder, decoder LoRA plus the projection head. `--decoder_max_length 200`
gives a budget of 328 (`main.py` adds `--multimodal_tokens`), which fits the
8-box maximum exactly — with zero headroom, so do not lengthen the captions in
`her2/captions.py` without rerunning `tools/check_her2_token_budget.py`.

Lower `--batch_size` to 8 if you hit OOM.

In [ ]:
RESULT_DIR = os.path.join(WORK, 'results', 'arm_a')
os.makedirs(RESULT_DIR, exist_ok=True)

subprocess.run([
    'python', 'main.py',
    '--dataset', 'her2', '--her2_dir', HER2_DIR,
    '--model_dir', MODELS, '--model_name', ENCODER, '--resume', RESUME,
    '--feature_cache_dir', CACHE,
    '--epochs', '30', '--batch_size', '16', '--num_workers', '2',
    '--decoder_max_length', '200', '--multimodal_tokens', '128',
    '--lr', '3e-4', '--cos_eta_min', '0.1',
    '--finetune_decoder', '--pool2x2', '--fix_enc',
    '--seed', '42', '--result_dir', RESULT_DIR,
], check=True)

In [ ]:
# Training curve + CSV/JSON summary for RESULTS_her2.md section 4
subprocess.run(['python', 'tools/plot_her2_training.py', '--result_dir', RESULT_DIR], check=True)

## 10. Evaluate

Three runs, and the third is not optional: the shuffled-image control. If it
scores close to the real evaluation, the model is not using the image and every
other number here is void. `RESULTS_her2.md` §6 fixed that rule before any of
this ran.

In [ ]:
import glob

checkpoints = sorted(glob.glob(os.path.join(RESULT_DIR, 'ep*.pt')))
assert checkpoints, f'no checkpoints in {RESULT_DIR}'
trained = checkpoints[-1]
print('Evaluating', trained)

common = ['--dataset', 'her2', '--her2_dir', HER2_DIR, '--model_dir', MODELS,
          '--model_name', ENCODER, '--multimodal_tokens', '128',
          '--decoder_max_length', '200', '--pool2x2', '--fix_enc', '--seed', '42']

# Arm A
subprocess.run(['python', 'main.py', '--evaluate', 'test', 'val',
                '--resume', trained, '--result_dir', RESULT_DIR] + common, check=True)

# Zero-shot baseline: the released checkpoint on the same split
baseline_dir = os.path.join(WORK, 'results', 'baseline')
os.makedirs(baseline_dir, exist_ok=True)
subprocess.run(['python', 'main.py', '--evaluate', 'test',
                '--resume', RESUME, '--result_dir', baseline_dir] + common, check=True)

# THE GATE: same checkpoint, same split, but every target paired with a different
# image. Writes to *_shuffled.csv/.pkl so it cannot overwrite the real evaluation.
subprocess.run(['python', 'main.py', '--evaluate', 'test', '--shuffle_images',
                '--resume', trained, '--result_dir', RESULT_DIR] + common, check=True)

In [ ]:
# Per-track breakdown -- the pooled number mixes pathologist polygons with
# threshold-derived boxes and has no single interpretation. Run it for the real
# evaluation AND the shuffled control, so the two are directly comparable.
for pkl in sorted(glob.glob(os.path.join(RESULT_DIR, 'eval_*_her2_ground_*.pkl'))):
    print('\n===', os.path.basename(pkl))
    subprocess.run(['python', 'tools/her2_eval_breakdown.py', '--pkl_path', pkl], check=True)

## 11. Before writing anything into RESULTS_her2.md

The reporting standard was fixed in advance (commit `9d0e98d`, before any run).
Re-read `RESULTS_her2.md` §6 and hold to it:

- The BCI track score is **not** a HER2 detection result. Its targets are boxes
  from our own DAB threshold; a high score means the model reproduced that
  heuristic, and it gets described that way without exception.
- If the shuffled-image control scores close to the real evaluation, stop — the
  model is not using the image and nothing else here means anything.
- If Arm A does not beat the zero-shot baseline, **that is the finding**, and it
  goes in as the headline rather than being buried.
- Never report a pooled figure across the two tracks.
- Nothing run with `--limit_samples` is a result.